In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [2]:
data = pd.read_csv('/kaggle/input/competitions/digit-recognizer/train.csv')

In [3]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
data = np.array(data)
m, n = data.shape
np.random.shuffle(data)

data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n] / 225.0

data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n] / 225.0

In [5]:
def init_params():
    W1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(0, Z)

def softmax(Z):
    Z = Z - np.max(Z, axis=0, keepdims=True)
    exp_Z = np.exp(Z)
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def deriv_ReLU(Z):
    return Z > 0

def back_prop(Z1, A1, Z2, A2, W2, X, Y):
  m = Y.size
  encoded_y = one_hot(Y)
  dZ2 = A2 - encoded_y
  dW2 = 1 / m * dZ2.dot(A1.T)
  db2 = 1 / m * np.sum(dZ2, axis=1, keepdims=True)
  dZ1 = W2.T.dot(dZ2) * deriv_ReLU(Z1)
  dW1 = 1 / m * dZ1.dot(X.T)
  db1 = 1 / m * np.sum(dZ1, axis=1, keepdims=True)
  return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2

In [6]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, iterations, alpha):
    W1, b1, W2, b2 = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = back_prop(Z1, A1, Z2, A2, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 50 == 0:
            print ("Iteration: ", i)
            print ("Accuracy: ", get_accuracy(get_predictions(A2), Y))
    return W1, b1, W2, b2

In [7]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 500, 0.1)

Iteration:  0
[8 1 1 ... 9 5 6] [2 3 5 ... 8 1 4]
Accuracy:  0.13651219512195123
Iteration:  50
[1 2 1 ... 8 1 0] [2 3 5 ... 8 1 4]
Accuracy:  0.44758536585365855
Iteration:  100
[1 2 6 ... 8 1 9] [2 3 5 ... 8 1 4]
Accuracy:  0.5789512195121951
Iteration:  150
[1 3 6 ... 8 1 4] [2 3 5 ... 8 1 4]
Accuracy:  0.6620243902439025
Iteration:  200
[1 3 5 ... 8 1 4] [2 3 5 ... 8 1 4]
Accuracy:  0.7119756097560975
Iteration:  250
[1 3 5 ... 8 1 4] [2 3 5 ... 8 1 4]
Accuracy:  0.7472926829268293
Iteration:  300
[1 3 5 ... 8 1 4] [2 3 5 ... 8 1 4]
Accuracy:  0.774390243902439
Iteration:  350
[1 3 5 ... 8 1 4] [2 3 5 ... 8 1 4]
Accuracy:  0.7926341463414635
Iteration:  400
[1 3 5 ... 8 1 4] [2 3 5 ... 8 1 4]
Accuracy:  0.8068780487804879
Iteration:  450
[1 3 5 ... 8 1 4] [2 3 5 ... 8 1 4]
Accuracy:  0.8182682926829268


In [18]:
def make_predictions(X, W1, b1, W2, b2):
  _, _, _, A2 = forward_prop(W1, b1, W2, b2, X)
  predictions = get_predictions(A2)
  return predictions


def test_prediction(index, W1, b1, W2, b2):
  current_image = X_train[:, index, None]

  prediction = make_predictions(current_image, W1, b1, W2, b2)
  label = Y_train[index]

  print('Prediction: ', prediction[0])
  print('Label: ', label)

  image = current_image.reshape(28, 28) * 255.0
  plt.gray()
  plt.imshow(image, interpolation='nearest')
  plt.show()

In [35]:
dev_predictions = make_predictions(X_dev, W1, b1, W2, b2)
get_accuracy(dev_predictions, Y_dev)

[0 2 4 6 9 8 6 1 5 1 1 9 6 5 4 7 0 7 8 2 2 2 7 7 0 8 1 7 7 4 4 0 2 3 1 9 4
 0 1 1 6 8 5 7 4 7 7 2 1 7 6 4 8 7 4 8 3 9 2 4 1 8 9 3 0 0 3 6 3 7 1 2 2 1
 2 2 3 1 8 9 3 5 8 6 3 1 2 3 1 6 1 5 4 8 4 7 6 2 2 5 8 4 2 6 6 5 0 2 5 0 7
 5 8 7 6 5 6 3 9 4 3 4 9 9 6 3 4 3 1 4 7 0 7 2 6 8 5 2 1 1 3 1 8 5 0 6 1 7
 2 2 2 4 7 8 9 2 2 9 8 9 7 3 7 2 9 6 8 8 1 0 3 8 4 8 5 6 3 6 7 3 0 3 4 9 2
 0 6 1 9 0 3 2 1 3 8 2 3 3 5 8 8 7 4 4 2 6 2 3 2 4 0 3 8 4 0 7 8 1 8 1 4 6
 2 4 9 6 9 4 5 4 4 0 6 8 4 6 6 9 8 4 0 2 3 1 2 1 1 7 9 8 1 2 0 1 9 6 3 9 1
 4 6 9 2 7 5 3 7 2 9 7 3 3 7 8 5 9 6 1 3 9 0 1 8 2 0 1 3 8 6 1 3 1 4 6 1 7
 1 0 0 4 6 4 6 7 7 9 8 6 4 3 7 6 1 5 4 3 0 1 7 6 9 5 1 6 0 5 8 0 5 1 0 6 9
 7 4 5 1 4 7 9 9 7 6 9 5 8 7 5 6 4 9 5 5 5 5 8 2 1 0 7 5 0 2 7 4 4 2 5 0 8
 0 7 9 3 0 1 3 9 9 6 9 5 0 2 4 4 9 0 9 7 6 7 2 0 2 4 3 8 9 1 1 6 0 7 0 2 5
 9 1 6 2 7 4 5 7 4 7 1 7 7 2 8 2 2 0 2 7 3 6 1 4 8 1 1 8 7 2 1 1 6 9 8 0 3
 2 9 0 4 6 3 6 8 0 8 7 2 2 3 1 7 1 0 3 5 1 8 9 6 3 9 9 1 1 7 8 1 2 7 2 5 9
 9 6 0 3 9 2 8 2 0 9 7 2 

np.float64(0.828)